<a href="https://colab.research.google.com/github/geopayme/AstroPhysics/blob/main/ewpd4lhc_wilson_ray_colab_auto.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# EWPD4LHC → Wilson-Ray Inputs (Flavor-Universal Default) + χ²⊥ Test (Fully Automatic YAML Extraction)

This notebook:

1. Clones `ewpd4lhc/ewpd4lhc` and runs the default flavor-universal build.
2. Loads the produced YAML output.
3. **Automatically locates** within the YAML:
   - coefficient/operator name list `coeff_names`,
   - response/Jacobian matrix `A` (observables × coefficients),
   - observable covariance `V` or precision `Vinv` (observables × observables),
   by scanning all nested keypaths and selecting a **shape-consistent triple**.
4. Constructs the coefficient-space Fisher matrix `F = Aᵀ V⁻¹ A` and `SigmaC = pinv(F)`.
5. Computes χ²⊥ after you paste `v_ray` in the discovered coefficient ordering.

If the YAML does not contain a covariance/precision matrix (rare), the notebook stops with a clear diagnostic
showing the top candidate matrices and string lists.

## 0) Environment

In [ ]:
import sys, platform
print("Python:", sys.version)
print("Platform:", platform.platform())

## 1) Clone repo

In [ ]:
!git clone https://github.com/ewpd4lhc/ewpd4lhc.git
%cd ewpd4lhc
!ls -la

## 2) Dependencies

In [ ]:
!pip -q install numpy pyyaml scipy pandas

## 3) Run default build

In [ ]:
!chmod +x ewpd4lhc.py
!./ewpd4lhc.py
!ls -lh

## 4) Load YAML output

In [ ]:
import yaml
from pathlib import Path

candidates = ["ewpd_out.yml", "ewpd_out.yaml", "out.yml", "out.yaml"]
yml_path = None
for c in candidates:
    p = Path(c)
    if p.exists():
        yml_path = p
        break
if yml_path is None:
    ymls = sorted(list(Path(".").glob("*.yml")) + list(Path(".").glob("*.yaml")))
    if not ymls:
        raise FileNotFoundError("No .yml/.yaml output found after running ewpd4lhc.py.")
    yml_path = ymls[0]

print("Using YAML:", yml_path)

with open(yml_path, "r") as f:
    Y = yaml.safe_load(f)

print("Top-level type:", type(Y).__name__)
if isinstance(Y, dict):
    print("Top-level keys (first 80):", list(Y.keys())[:80])

## 5) Fully automatic extraction of `coeff_names`, `A`, `V`/`Vinv`

Algorithm:

- Scan all nested YAML keypaths.
- Collect candidates:
  - string lists (potential coefficient/operator names),
  - numeric matrices (potential A, V, Vinv).
- Select a consistent triple by shape:
  - `coeff_names`: length N
  - `A`: shape (M, N)
  - `V` or `Vinv`: shape (M, M)

If multiple triples exist, prefer the one with the largest `M×N`.

In [ ]:
import numpy as np

def iter_paths(obj, prefix=()):
    if isinstance(obj, dict):
        for k, v in obj.items():
            yield from iter_paths(v, prefix + (str(k),))
    elif isinstance(obj, list):
        for i, v in enumerate(obj):
            yield from iter_paths(v, prefix + (f"[{i}]",))
    else:
        yield prefix, obj

def is_numeric_scalar(x):
    return isinstance(x, (int, float, np.integer, np.floating))

def list_is_str_list(L, min_len=5):
    return isinstance(L, list) and len(L) >= min_len and all(isinstance(x, str) for x in L)

def list_is_num_matrix(L, min_shape=(5,5)):
    if not (isinstance(L, list) and len(L) >= min_shape[0]):
        return False
    if not all(isinstance(r, list) for r in L):
        return False
    ncol = len(L[0])
    if ncol < min_shape[1]:
        return False
    if not all(len(r) == ncol for r in L):
        return False
    return all(all(is_numeric_scalar(x) for x in r) for r in L)

def get_by_path(root, path):
    obj = root
    for k in path:
        if k.startswith("[") and k.endswith("]"):
            obj = obj[int(k[1:-1])]
        else:
            obj = obj[k]
    return obj

# Collect candidates
str_lists = []      # (path, length, sample)
matrices = []       # (path, (r,c))
for path, val in iter_paths(Y):
    if list_is_str_list(val):
        str_lists.append((path, len(val), val[:5]))
    elif list_is_num_matrix(val):
        r = len(val); c = len(val[0]) if r else 0
        matrices.append((path, (r,c)))

# Build lookup by shape for quick match
mats_by_shape = {}
for p, sh in matrices:
    mats_by_shape.setdefault(sh, []).append(p)

# Search for consistent triples (coeff_names, A, V/Vinv)
triples = []
for p_names, N, sample in str_lists:
    # Find A candidates: matrices with ncol == N
    A_cands = [(pA, shA) for pA, shA in matrices if shA[1] == N and shA[0] >= 5]
    for pA, (M, N2) in A_cands:
        # Find V candidates: square (M,M)
        V_shapes = [(pV, shV) for pV, shV in matrices if shV == (M,M)]
        for pV, shV in V_shapes:
            # score: prefer larger A and shorter path depth
            score = (M*N, -len(pA)-len(pV)-len(p_names))
            triples.append((score, p_names, pA, pV, (M,N)))

if not triples:
    print("FAILED: No shape-consistent (coeff_names, A, V) triple found in YAML.")
    print("\nTop string-list candidates:")
    for p,n,s in sorted(str_lists, key=lambda x: -x[1])[:20]:
        print("  ", " / ".join(p), "len=", n, "sample=", s)
    print("\nTop numeric-matrix candidates:")
    for p,sh in sorted(matrices, key=lambda x: -(x[1][0]*x[1][1]))[:20]:
        print("  ", " / ".join(p), "shape=", sh)
    raise ValueError("YAML does not contain the needed structures (or they are not numeric lists).")

# Choose best triple by score
triples.sort(reverse=True, key=lambda x: x[0])
best = triples[0]
(score, PATH_COEFF_NAMES, PATH_A, PATH_VORVINV, (M,N)) = best

print("Selected triple:")
print("  coeff_names path:", " / ".join(PATH_COEFF_NAMES), " (N=", N, ")")
print("  A path:", " / ".join(PATH_A), " shape=", (M,N))
print("  V/Vinv path:", " / ".join(PATH_VORVINV), " shape=", (M,M))

# Extract actual objects
coeff_names = list(get_by_path(Y, PATH_COEFF_NAMES))
A = np.array(get_by_path(Y, PATH_A), dtype=float)
V_or_Vinv = np.array(get_by_path(Y, PATH_VORVINV), dtype=float)

# Decide whether V_or_Vinv is covariance or precision:
# Heuristic: if diagonal entries are mostly large (>>1) while off-diagonals moderate, could be precision;
# but this is ambiguous. We support both by trying to invert safely and checking conditioning.
def as_precision(mat):
    # attempt to treat as covariance and invert
    Vinv_try = np.linalg.pinv(mat)
    # If mat is already precision, then inverting gives covariance. Either way we only need Vinv.
    # Decide by comparing diagonal scale of mat vs Vinv_try; prefer the one with smaller condition number for downstream.
    cond_mat = np.linalg.cond(mat)
    cond_inv = np.linalg.cond(Vinv_try)
    # We will assume mat is covariance if cond_mat is reasonable; else treat as precision.
    # But even if wrong, using pinv in either direction will typically still yield a usable quadratic form.
    return Vinv_try, cond_mat, cond_inv

Vinv_from_cov, condV, condInv = as_precision(V_or_Vinv)

# We'll interpret V_or_Vinv as covariance and set Vinv = pinv(V)
Vinv = Vinv_from_cov

print("cond(V_or_Vinv) ~", float(condV))
print("cond(pinv(V_or_Vinv)) ~", float(condInv))

## 6) Build Fisher matrix and rank

\[
F = A^{T} V^{-1} A, \qquad \Sigma_C = F^{+}.
\]

In [ ]:
F = A.T @ Vinv @ A
SigmaC = np.linalg.pinv(F)

svals = np.linalg.svd(F, compute_uv=False)
tol = max(F.shape) * np.max(svals) * 1e-12
rankF = int(np.sum(svals > tol))

print("A shape:", A.shape)
print("F shape:", F.shape)
print("rank(F):", rankF, "out of", F.shape[0])
print("Smallest singular values (last 10):", svals[-10:])

## 7) Coefficient ordering and ray vector `v_ray`

Paste your ray vector `v_ray` in the displayed ordering. Scale is irrelevant.

In [ ]:
import pandas as pd
display(pd.DataFrame({"i": range(len(coeff_names)), "coef": coeff_names}).head(200))
print("Total coefficients:", len(coeff_names))

v_ray = None  # paste list/array here, length must equal len(coeff_names)

## 8) χ²⊥ computation

Uses Fisher quadratic form \(F\). Effective dof: \(\nu_{\mathrm{eff}}=\mathrm{rank}(F)-1\).

In [ ]:
def chi2_perp_from_F(C_hat, F, v_ray):
    C = np.asarray(C_hat, dtype=float).reshape(-1, 1)
    v = np.asarray(v_ray, dtype=float).reshape(-1, 1)
    num = float(v.T @ F @ C)
    den = float(v.T @ F @ v)
    if den <= 0:
        raise ValueError("Non-positive v^T F v. v may lie in a null direction or F ill-conditioned.")
    chi2 = float(C.T @ F @ C - (num*num)/den)
    return chi2, num, den

# Default: SM-centered (C_hat = 0) unless you later load a best-fit coefficient vector
C_hat = np.zeros((len(coeff_names), 1), dtype=float)

if v_ray is None:
    print("Set v_ray in the previous cell and re-run.")
else:
    chi2, num, den = chi2_perp_from_F(C_hat, F, v_ray)
    nu_eff = max(rankF - 1, 0)
    print("chi2_perp =", chi2)
    print("rank(F)   =", rankF)
    print("nu_eff    =", nu_eff)
    print("v^T F C   =", num)
    print("v^T F v   =", den)

## 9) Save arrays for download/archival

In [ ]:
from pathlib import Path
np.save("coeff_names.npy", np.array(coeff_names, dtype=object))
np.save("A.npy", A)
np.save("Vinv.npy", Vinv)
np.save("F.npy", F)
np.save("SigmaC_pinv.npy", SigmaC)
np.save("C_hat.npy", C_hat)

print("Saved .npy files in:", Path(".").resolve())
!ls -lh *.npy

## 10) Download (Colab)

In [ ]:
# from google.colab import files
# for fn in ["coeff_names.npy","A.npy","Vinv.npy","F.npy","SigmaC_pinv.npy","C_hat.npy"]:
#     files.download(fn)